# 01 — Extração de características (Kaggle, CPU)

Gera `features/slices/*.parquet` para a amostra do TP1 (3.000 exames, 1 corte a cada 2, ≈ 370 mil cortes).

**Custo real medido:** ≈ 8,5 s por exame em 4 CPUs, ou seja ≈ 7 h de CPU para a amostra toda.
Por isso a extração é **divisível em shards**: rode uma cópia deste notebook por shard
(`SHARD` = 0..N_SHARDS-1) em sessões paralelas (o Kaggle permite até 5 sessões de CPU ao mesmo
tempo). Os shards são disjuntos e, juntos, cobrem todos os exames. Dentro de uma sessão o
processo é retomável: blocos já gravados são pulados.

Proteções: cada corte tem limite de tempo (`preprocess.slice_timeout_s`) e cada exame roda em
processo separado com limite duro (`exam_timeout_s`); um exame travado vira linha de erro
registrada, não um bloco parado (isso aconteceu duas vezes no Kaggle).

Ao final: **Save Version → Save & Run All**. A saída (`/kaggle/working/features`) vira um Dataset
que o notebook 02 usa como Input; some todas as saídas dos shards como Inputs do notebook 02.

In [ ]:
# === Setup (Kaggle) ==========================================================
# Pré-requisitos: aceitar as regras da competição "RSNA STR Pulmonary Embolism Detection"
# e adicioná-la como Input deste notebook (+ Add Input -> Competitions).
import glob, os, sys, subprocess, shutil
REPO_URL = "https://github.com/silvarawenne/ap2-topicos.git"
REPO_BRANCH = "claude/peaceful-curie-mqrljz"   # troque para main após o merge
REPO = "/kaggle/working/ap2-topicos"

if not os.path.exists(REPO):
    # opção sem Internet: o repositório enviado como Dataset (tem configs/config.yaml na raiz)
    hits = glob.glob("/kaggle/input/**/configs/config.yaml", recursive=True)
    if hits:
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), REPO)
    else:  # exige "Internet on" nas configurações do notebook
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-kaggle.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
# os dados da competição são achados sozinhos (ou defina PE_DATA_ROOT)
hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
if hits:
    os.environ.setdefault("PE_DATA_ROOT", os.path.dirname(hits[0]))
print("repo:", REPO, "| dados:", os.environ.get("PE_DATA_ROOT"))


In [ ]:
# Escolha o shard desta cópia (N_SHARDS=1 roda tudo numa sessão só, ≈ 7 h)
SHARD, N_SHARDS = 0, 1
os.environ["PE_FEATURES_DIR"] = "/kaggle/working/features"
!python scripts/01_make_split_and_sample.py
!python -W ignore scripts/02_extract_features.py --shard {SHARD} --n-shards {N_SHARDS}

In [ ]:
import pandas as pd
files = sorted(glob.glob("/kaggle/working/features/slices/*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=["StudyInstanceUID", "status"]) for f in files])
print(len(files), "blocos |", len(df), "cortes |", df.StudyInstanceUID.nunique(), "exames")
print(df.status.str.slice(0, 60).value_counts().head())

In [ ]:
# Partição e amostra usadas na extração acompanham a saída (o notebook 02 as lê de lá)
!cp -r splits /kaggle/working/features/ && du -sh /kaggle/working/features